In [1]:
from google.colab import files
uploaded=files.upload()


Saving attendace.csv to attendace.csv
Saving employees.csv to employees.csv
Saving tasks.csv to tasks.csv


In [2]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("EmployeeAttendanceAnalysis") \
    .getOrCreate()

df = spark.read.csv(
    "attendapce.csv",
    header=True,
    inferSchema=True
)

df = df.fillna({
    "clock_in":"09:00"
})

df = df.withColumn(
    "clock_in",
    to_timestamp("clock_in","HH:mm")
)

df = df.withColumn(
    "clock_out",
    to_timestamp("clock_out","HH:mm")
)

df = df.withColumn(
    "work_hours",
    (unix_timestamp("clock_out") -
     unix_timestamp("clock_in")) / 3600
)

df = df.withColumn(
    "productivity_score",
    col("tasks_completed") / col("work_hours")
)

late_logins = df.filter(
    hour("clock_in") > 9
)

print("Late Logins")
late_logins.show()

absentees = df.filter(
    col("work_hours") < 6
)

print("Frequent Absentees")
absentees.show()

dept_summary = df.groupBy("department").agg(
    avg("work_hours").alias("avg_work_hours"),
    avg("productivity_score").alias("avg_productivity")
)

print("Department Summary")
dept_summary.show()

Late Logins
+-----------+-------------+----------+-------------------+-------------------+---------------+----------+------------------+
|employee_id|employee_name|department|           clock_in|          clock_out|tasks_completed|work_hours|productivity_score|
+-----------+-------------+----------+-------------------+-------------------+---------------+----------+------------------+
|        103|        Rahul|   Finance|2026-05-14 10:00:00|2026-05-14 15:00:00|              2|       5.0|               0.4|
|        106|        Meera|        HR|2026-05-14 10:30:00|2026-05-14 16:00:00|              3|       5.5|0.5454545454545454|
|        108|        Divya|   Finance|2026-05-14 11:00:00|2026-05-14 15:30:00|              1|       4.5|0.2222222222222222|
+-----------+-------------+----------+-------------------+-------------------+---------------+----------+------------------+

Frequent Absentees
+-----------+-------------+----------+-------------------+-------------------+---------------